# Gene-set analyses of the comorbidity sub-graphs

The gene-set counterpart of `4_10` and `4_15`: instead of drawing the sub-graph around each interface protein, we turn it into a **gene set** and confront it with the bulk RNA-seq DE gene lists of `4_00`.

Two analyses, on two pairings.

**The analyses.** *GOAT* tests the sub-graph's gene set for enrichment against the whole effect-size ranking of a gene list (the R `goat` package, via `rpy2`). The *intersection* is its naive counterpart — how much of the gene set is simply among the genes the list marks significant. They want different namespaces: GOAT matches the gene lists' `gene` column, which is the HGNC dataset's `entrez_id`, so its gene sets are `ncbigene`; the intersection matches the `symbol` column, so its gene sets are `hgnc.symbol`.

**The pairings.** COVID → PD, whose downstream side is the PD activity-flow maps, and COVID → AD, whose downstream side is the **AD BEL knowledge graph** — the same two `4_10` and `4_15` draw, with the same collection names, walk and `MAX_LEVELS`. The AD side used to be out of reach here: BEL nodes carried only `uniprot` annotations, and they were labelled `BELModelElement` but not `ModelElement`, which is what the annotation helpers in `commute_dm.queries` match — so they returned nothing for a BEL node, silently. `2_00` now writes `hgnc.symbol`, `ncbigene`, `hgnc` and `uniprot` on the KG's HGNC-encoded proteins and labels its nodes `ModelElement`, which is all it took.

Nothing here writes to the database, and nothing here loads a map: gene sets need no drawing material, so this is a matter of seconds rather than the couple of minutes `4_10`/`4_15` spend hydrating.

In [1]:
%store -r

In [2]:
import commute_dm.core
import commute_dm.utils
import credentials
import momapy_kb.lpg.backends.neo4j
import momapy_kb.lpg.session

In [3]:
backend = momapy_kb.lpg.backends.neo4j.Neo4jBackend(
    hostname=credentials.NEO4J_URI,
    username=credentials.NEO4J_USERNAME,
    password=credentials.NEO4J_PASSWORD,
    notifications_min_severity="off",
)
session = momapy_kb.lpg.session.Session(backend)

## Parameters

One entry per pairing, holding exactly what `4_10` and `4_15` hold in their own parameter cells — the collection names, the interface to join over and the levels to walk — plus the output directory of each analysis and mode.

`MAX_LEVELS` differs between the two for the reason `4_15` gives: the AD influence graph is far denser than an activity-flow map, so a level means much more there. `WITH_SUBUNITS` now means the same thing on both sides — a CellDesigner complex's subunits, and a BEL complex's or composite's members. A BEL **activity** is not a subunit and is always followed: `act(p(X))` is X in another form, the way an active CellDesigner species is still that species.

In [4]:
MIN_N_NODES = 5
WITH_SUBUNITS = True
P_VALUE_CUTOFF = 0.05
MODES = ["upstream", "downstream", "upstream_and_downstream"]

PAIRINGS = [
    {
        "name": "COVID -> PD",
        # The interface is the three-way one, as in `4_10`: a protein has to be
        # shared with the AD KG as well to be looked at here.
        "upstream_collection_name": "COVID_DM_CD_AF",
        "downstream_collection_name": "PD_DM_CD_AF",
        "interface_collection_names": (
            "COVID_DM_CD_AF",
            "PD_DM_CD_AF",
            "AD_KG_BEL",
        ),
        "max_levels": [2, 3, 4, 5, 6],
        "goat_dir_paths": {
            "upstream": INTERFACE_GOAT_UPSTREAM_ANALYSIS_DIR,
            "downstream": INTERFACE_GOAT_DOWNSTREAM_ANALYSIS_DIR,
            "upstream_and_downstream": (
                INTERFACE_GOAT_UPSTREAM_AND_DOWNSTREAM_ANALYSIS_DIR
            ),
        },
        "intersection_dir_paths": {
            "upstream": INTERFACE_INTERSECTION_UPSTREAM_ANALYSIS_DIR,
            "downstream": INTERFACE_INTERSECTION_DOWNSTREAM_ANALYSIS_DIR,
            "upstream_and_downstream": (
                INTERFACE_INTERSECTION_UPSTREAM_AND_DOWNSTREAM_ANALYSIS_DIR
            ),
        },
    },
    {
        "name": "COVID -> AD",
        # Two-way, as in `4_15`: the pairing *is* COVID and the AD KG.
        "upstream_collection_name": "COVID_DM_CD_AF",
        "downstream_collection_name": "AD_KG_BEL",
        "interface_collection_names": (
            "COVID_DM_CD_AF",
            "AD_KG_BEL",
        ),
        "max_levels": [1, 2, 3],
        "goat_dir_paths": {
            "upstream": INTERFACE_AD_GOAT_UPSTREAM_ANALYSIS_DIR,
            "downstream": INTERFACE_AD_GOAT_DOWNSTREAM_ANALYSIS_DIR,
            "upstream_and_downstream": (
                INTERFACE_AD_GOAT_UPSTREAM_AND_DOWNSTREAM_ANALYSIS_DIR
            ),
        },
        "intersection_dir_paths": {
            "upstream": INTERFACE_AD_INTERSECTION_UPSTREAM_ANALYSIS_DIR,
            "downstream": INTERFACE_AD_INTERSECTION_DOWNSTREAM_ANALYSIS_DIR,
            "upstream_and_downstream": (
                INTERFACE_AD_INTERSECTION_UPSTREAM_AND_DOWNSTREAM_ANALYSIS_DIR
            ),
        },
    },
]

## The interface and the influence structure

For each pairing we compute its interface and load what the walk needs. `load_gene_set_inputs` is `load_submap_inputs` minus the drawing material: the influence structure of the CellDesigner collections, a BEL downstream side's influence-graph projection merged into it, and the **seed expansion** — the UniProt annotations that define the interface sit on the KG's `p(HGNC:X)` nodes while BEL keeps the causal wiring on `act(p(HGNC:X))`, so a downstream seed is widened to its activity forms. It is empty for a CellDesigner downstream side, where the annotated node *is* the wired node.

In [5]:
for pairing in PAIRINGS:
    pairing["interface"] = commute_dm.core.get_interface(
        session, pairing["interface_collection_names"]
    )
    (
        pairing["influences"],
        pairing["seed_expansion"],
    ) = commute_dm.core.load_gene_set_inputs(
        session,
        pairing["upstream_collection_name"],
        pairing["downstream_collection_name"],
    )
    # Once per pairing, not once per mode: this is a query per interface protein.
    pairing["display_names"] = commute_dm.core.get_interface_display_names(
        session, pairing["interface"]
    )

{
    pairing["name"]: {
        "n_interface": len(pairing["interface"]),
        "n_seeds_expanded": len(pairing["seed_expansion"]),
    }
    for pairing in PAIRINGS
}

{'COVID -> PD': {'n_interface': 127, 'n_seeds_expanded': 0},
 'COVID -> AD': {'n_interface': 189, 'n_seeds_expanded': 79}}

## GOAT enrichment

For each pairing, each mode and each level, the gene set of the selection around every interface protein is tested against every gene list. Output is one CSV per gene list per level, plus a `summary.csv` listing, per interface protein, the levels at which it came out significant — ordered by how often it did.

The three modes are the three ways to read a comorbidity sub-graph: what the interface protein is downstream of in COVID (`upstream`), what it drives in the comorbid disease (`downstream`), and both at once.

`MIN_N_NODES` applies to **both** directions, so an interface protein is skipped at a level unless each side reaches at least that many nodes — which is why the counts above are much larger than the number of proteins actually tested.

In [ ]:
for pairing in PAIRINGS:
    for mode in MODES:
        output_dir_path = pairing["goat_dir_paths"][mode]
        commute_dm.utils.remake_dir(output_dir_path)
        print(f"GOAT: {pairing['name']}, {mode} -> {output_dir_path}")
        commute_dm.core.make_goat_analysis_from_interface(
            session,
            pairing["interface"],
            pairing["influences"],
            GOAT_GENE_LISTS_BUILD_DIR,
            output_dir_path,
            upstream_collection_name=pairing["upstream_collection_name"],
            downstream_collection_name=pairing["downstream_collection_name"],
            mode=mode,
            max_levels=pairing["max_levels"],
            with_subunits=WITH_SUBUNITS,
            p_value_cutoff=P_VALUE_CUTOFF,
            min_n_nodes=MIN_N_NODES,
            downstream_node_id_expansion=pairing["seed_expansion"],
            display_names=pairing["display_names"],
        )

GOAT: COVID -> PD, upstream -> ../../build/results/interface/analysis/goat_upstream
GOAT: COVID -> PD, downstream -> ../../build/results/interface/analysis/goat_downstream


## Intersection with the significant genes

The same selections, read the naive way: how much of each sub-graph's gene set is among the genes a list marks significant. Gene sets are `hgnc.symbol` here rather than `ncbigene`, because that is what the gene lists' `symbol` column holds.

This is not a test — it has no null model and no p-value, and a big sub-graph will overlap more by construction, which is exactly what GOAT above corrects for. It is here because it says *which* genes, where GOAT only says whether. The `summary.csv` orders interface proteins by the mean `|intersection| / |gene set|` over the levels.

In [ ]:
for pairing in PAIRINGS:
    for mode in MODES:
        output_dir_path = pairing["intersection_dir_paths"][mode]
        commute_dm.utils.remake_dir(output_dir_path)
        print(f"intersection: {pairing['name']}, {mode} -> {output_dir_path}")
        commute_dm.core.make_intersection_analysis_from_interface(
            session,
            pairing["interface"],
            pairing["influences"],
            GOAT_GENE_LISTS_BUILD_DIR,
            output_dir_path,
            upstream_collection_name=pairing["upstream_collection_name"],
            downstream_collection_name=pairing["downstream_collection_name"],
            mode=mode,
            max_levels=pairing["max_levels"],
            with_subunits=WITH_SUBUNITS,
            min_n_nodes=MIN_N_NODES,
            downstream_node_id_expansion=pairing["seed_expansion"],
            display_names=pairing["display_names"],
        )

## What came out

The head of each `summary.csv`: the interface proteins whose sub-graph was most often significant (GOAT) or most heavily overlapping (intersection), per pairing and mode.

Every output table is keyed by UniProt accession (`identifier`) with the HGNC symbol beside it (`display_name`) — the same name `4_10`/`4_15` give the protein's map directory. The accession stays because it is what joins the collections and what is guaranteed unique: `BBC3` is the symbol of two accessions in the current data, and `get_interface_display_names` disambiguates those as `BBC3_<accession>` rather than letting two proteins collapse into one row.

In [ ]:
import os.path

import pandas

for pairing in PAIRINGS:
    for analysis, dir_paths in (
        ("goat", pairing["goat_dir_paths"]),
        ("intersection", pairing["intersection_dir_paths"]),
    ):
        for mode in MODES:
            summary_file_path = os.path.join(dir_paths[mode], "summary.csv")
            if not os.path.exists(summary_file_path):
                continue
            summary_df = pandas.read_csv(summary_file_path, index_col=0)
            print(
                f"\n=== {pairing['name']} / {analysis} / {mode} "
                f"({len(summary_df)} proteins)"
            )
            display(summary_df.head(10))